In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

Section A - Random Forest Implementation
In this part, we implement Random Forest from scratch for two tasks:

Classification - predicting the vehicle condition
Regression - predicting the vehicle price
The implementation is based on multiple decision trees. For classification, the final prediction is determined by majority voting. For regression, the final prediction is the average of all tree predictions, as required in the assignment.

Random Forest - General Idea
Random Forest is an ensemble learning method that combines multiple decision trees. Each tree is trained on a different bootstrap sample of the training data. This helps reduce the dependence on a single tree and improves generalization.

At prediction time:

In classification, each tree predicts a class, and the forest returns the most common class.
In regression, each tree predicts a numeric value, and the forest returns the average prediction.
Assignment Constraints
The implementation follows the assignment requirements:

If two features produce the exact same split score (Gini Gain in classification or SSR reduction in regression), the feature that comes first alphabetically must be selected.

The parameter min_samples_leaf is enforced. Any split that creates a leaf with fewer than 5 samples is discarded.

In Random Forest regression, the final prediction at test time is computed as the average of the predictions of all trees.

These rules must be respected in every tree used inside the forest.

In [2]:
# Class for one node in a tree
class Node:
    def __init__(
        self,
        impurity=None,
        num_samples=None,
        prediction=None,
        num_samples_per_class=None,
        feature_index=None,
        threshold=None,
        left=None,
        right=None
    ):
        # impurity at this node
        self.impurity = impurity

        # number of samples in this node
        self.num_samples = num_samples

        # stored prediction:
        self.prediction = prediction

        # class counts (used in classification if needed)
        self.num_samples_per_class = num_samples_per_class

        # Split rule
        self.feature_index = feature_index
        self.threshold = threshold

        # Children
        self.left = left
        self.right = right

    def is_leaf_node(self):
        return self.left is None and self.right is None

In [3]:
def gini(y):
    """
    Compute the Gini impurity of a target vector y.
    """
    n = len(y)
    
    # Empty node
    if n == 0:
        return 0

    # Count samples in each class
    _, counts = np.unique(y, return_counts=True)

    # Convert counts to probabilities
    probabilities = counts / n

    # Apply Gini formula
    return 1 - np.sum(probabilities ** 2)

In [2]:
# Helper functions for Random Forest

import numpy as np


def bootstrap_sample(X, y):
    """
    Create a bootstrap sample (sampling with replacement).
    The sample has the same size as the original dataset.
    Works with pandas DataFrame and Series.
    """
    n_samples = X.shape[0]

    # sample row indices with replacement
    indices = np.random.choice(n_samples, size=n_samples, replace=True)

    X_resampled = X.iloc[indices]
    y_resampled = y.iloc[indices]

    return X_resampled, y_resampled

def majority_vote(predictions):
    """
    Return the most common class among predictions.
    Used in classification.
    """
    values, counts = np.unique(predictions, return_counts=True)
    return values[np.argmax(counts)]


def average_predictions(predictions):
    """
    Return the average of predictions.
    Used in regression.
    """
    return np.mean(predictions)